#### roja y moda

In [ ]:
"""
Histogramas de distribución de cursadas por materia, según número de cuatrimestre.

Reutiliza:
- cargar_base_siu (data_loader.py) para levantar el dataset crudo.
- El mismo criterio de ordinalidad de step_nro usado en graph_engine.generate_nodes_and_edges
  (ordenar por [persona, date_columns] y hacer cumcount por persona), pero aplicado
  a TODAS las cursadas (no solo las aprobadas), ya que acá interesa "en qué cuatrimestre
  cursó" y no el estado acumulado de aprobación.

Eje X: step_nro (número de cuatrimestre de esa persona, 1ro, 2do, 3ro...)
Eje Y: cantidad de Estudiantes Únicas que cursaron esa materia en ese step_nro.
La barra correspondiente al cuatrimestre sugerido (según el plan CSV) se resalta en rojo.

Nota: se cuenta "cantidad de personas", no "cantidad de cursadas". Si una persona
recursa la misma materia dos veces en el mismo step_nro (no debería pasar dado que
step_nro es por persona-fecha), igual se cuenta una sola vez (nunique). Si una persona
cursa la misma materia en dos step_nro distintos (recursada en otro cuatrimestre),
aparece una vez en cada barra correspondiente -- es el comportamiento esperado para
un histograma de "en qué cuatrimestre se cursó" (no "aprobó").
"""

from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from visualization_prod.core.data_loader import cargar_base_siu, load_plan_from_directory, build_unified_catalog, get_first_year_materias
from visualization_prod.core.utils import get_camada
from visualization_prod.layout import ROJO,AZUL


def compute_step_nro_por_cursada(
    cursadas_por_alumno: pd.DataFrame,
    camada: list[str],
    persona_column: str = "persona",
    date_columns: list[str] | None = None,
) -> pd.DataFrame:
    """Agrega columna step_nro a cursadas_por_alumno: mismo criterio ordinal que
    generate_nodes_and_edges (cumcount agrupado por persona, ordenado por fecha),
    pero calculado sobre TODAS las cursadas (no solo aprobadas), para poder
    ubicar en qué cuatrimestre ocurrió cada cursada individual, sin importar
    si la materia en cuestión fue o no aprobada en ese intento.
    """
    date_columns = date_columns or ["anio_academico", "periodo"]
    df = cursadas_por_alumno[cursadas_por_alumno[persona_column].isin(camada[persona_column])].copy()

    step_order = (
        df[[persona_column] + date_columns]
        .drop_duplicates()
        .sort_values([persona_column] + date_columns)
    )
    step_order["step_nro"] = step_order.groupby(persona_column).cumcount() + 1

    df = df.merge(step_order, on=([persona_column] + date_columns))

    df = df[~(df["resultado"].isin({"U"}))]#saco los ausentes]
    return df


def build_materia_step_counts(
    df_con_step: pd.DataFrame,
    codigo_materia: str,
    persona_column: str = "persona",
    materia_column: str = "codigo_materia",
) -> pd.DataFrame:
    """Cuenta Estudiantes Únicas por step_nro para una materia dada.
    Devuelve DataFrame con columnas [step_nro, cantidad_personas], incluyendo
    steps intermedios sin datos (cantidad_personas=0), para que el histograma
    no salte visualmente entre barras."""
    subset = df_con_step[df_con_step[materia_column] == codigo_materia  ] 

    if subset.empty:
        return pd.DataFrame(columns=["step_nro", "cantidad_personas"])

    counts = (
        subset.groupby("step_nro")[persona_column]
        .nunique()
        .rename("cantidad_personas")
        .reset_index()
    )

    # completar steps faltantes con 0, entre el min y max step_nro observado para esta materia
    full_range = pd.DataFrame({"step_nro": range(int(counts["step_nro"].min()), int(counts["step_nro"].max()) + 1)})
    counts = full_range.merge(counts, on="step_nro", how="left")
    counts["cantidad_personas"] = counts["cantidad_personas"].fillna(0).astype(int)
    return counts


def load_plan_csv(plan_csv_path: Path) -> pd.DataFrame:
    """Carga el CSV de plan de estudios (formato legacy: codigo_materia, nombre_oficial,
    nomenclatura, cuatrimestre_sugerido). No usa data_loader.load_study_plan porque ese
    exige el esquema nuevo (nombreLargo/nombreCorto/cuatrimestreSugerido) y este CSV
    trae el esquema viejo -- son fuentes/formatos distintos, no se fuerza conversión."""
    plan = pd.read_csv(plan_csv_path)
    required = {"codigo_materia", "nombre_oficial", "cuatrimestre_sugerido"}
    missing = required - set(plan.columns)
    if missing:
        raise ValueError(f"Al plan CSV le faltan columnas: {missing}")
    return plan


def compute_moda_y_distancia(
    counts: pd.DataFrame,
    cuatri_sugerido: int | None,
) -> tuple[int | None, int | None]:
    """Calcula el step_nro donde se concentra la mayor cantidad de personas (moda)
    y la distancia con signo respecto al cuatrimestre sugerido (moda - sugerido).
    Distancia positiva: se cursa después de lo sugerido. Negativa: antes.
    Devuelve (moda, distancia); ambos None si no hay datos o no hay cuatri_sugerido."""
    if counts.empty:
        return None, None

    moda = int(counts.loc[counts["cantidad_personas"].idxmax(), "step_nro"])
    distancia = (moda - cuatri_sugerido) if cuatri_sugerido is not None else None
    return moda, distancia


def plot_histograma_materia(
    counts: pd.DataFrame,
    codigo_materia: str,
    nombre_largo: str | None = None,
    cuatri_sugerido: int | None = None,
    output_path: Path | None = None,
) -> plt.Figure:
    """Genera el histograma de barras para una materia. Si se pasa cuatri_sugerido,
    esa barra puntual se pinta de rojo para resaltar el cuatrimestre curricularmente
    esperado según el plan, contra la distribución real observada. Debajo del título
    se agrega la distancia con signo entre la moda observada y el cuatri sugerido."""
    fig, ax = plt.subplots(figsize=(8, 5))

    colors = [
        ROJO if step == cuatri_sugerido else AZUL
        for step in counts["step_nro"]
    ]
    ax.bar(counts["step_nro"], counts["cantidad_personas"], color=colors, edgecolor="black")

    titulo = nombre_largo if nombre_largo else codigo_materia
    ax.set_title(f"Distribución de cursadas por cuatrimestre — {titulo}", fontsize=12, pad=28)

    _, distancia = compute_moda_y_distancia(counts, cuatri_sugerido)
    if distancia is not None:
        signo = "+" if distancia >= 0 else ""
        ax.text(0.5, 1.05, f"Δ moda vs. sugerido: {signo}{distancia}",
                 transform=ax.transAxes, ha="center", fontsize=9, color="#555555")

    ax.set_xlabel("Número de Cuatrimestre Cursado (step_nro)")
    ax.set_ylabel("Cantidad de Estudiantes Únicas")
    ax.set_xticks(counts["step_nro"])
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    if cuatri_sugerido is not None:
        ax.bar(0, 0, color=ROJO, label=f"Cuatrimestre sugerido ({cuatri_sugerido})")
        ax.legend(loc="upper right", fontsize=9)

    fig.tight_layout()
    if output_path:
        fig.savefig(output_path, dpi=150)
    return fig


def _plot_en_grilla(
    df_con_step: pd.DataFrame,
    plan: pd.DataFrame,
    codigos: list[str],
    n_columnas: int,
    output_path: Path,
    titulo_plot:str,
    persona_column: str = "persona",
    materia_column: str = "codigo_materia",
) -> plt.Figure:
    """Helper compartido: arma una grilla de subplots (n_columnas fijas, filas
    según cantidad de materias) con un histograma por materia, resaltando en
    rojo el cuatrimestre sugerido. Subplots sobrantes (si la cantidad de
    materias no completa la última fila) se ocultan en vez de dejarse vacíos."""
    nombre_map = plan.set_index("codigo_materia")["nombre_oficial"].to_dict()
    cuatri_map = plan.set_index("codigo_materia")["cuatrimestre_sugerido"].to_dict()

    n_materias = len(codigos)
    n_filas = -(-n_materias // n_columnas)  # ceil division

    fig, axes = plt.subplots(n_filas, n_columnas, figsize=(5 * n_columnas, 4 * n_filas))
    axes = axes.flatten() if n_materias > 1 else [axes]

    # for ax, codigo in zip(axes, codigos):
    #     counts = build_materia_step_counts(df_con_step, codigo, persona_column, materia_column)
    #     cuatri_sugerido = cuatri_map.get(codigo)
    #     nombre_largo = nombre_map.get(codigo, codigo)

    #     if counts.empty:
    #         ax.set_title(f"{nombre_largo} (sin datos)", fontsize=10)
    #         ax.axis("off")
    #         continue

    #     colors = [
    #         ROJO if step == cuatri_sugerido else AZUL
    #         for step in counts["step_nro"]
    #     ]
    #     ax.bar(counts["step_nro"], counts["cantidad_personas"], color=colors, edgecolor="black")

    #     _, distancia = compute_moda_y_distancia(counts, cuatri_sugerido)
    #     if distancia is not None:
    #         signo = "+" if distancia >= 0 else ""
    #         titulo_completo = f"{nombre_largo}\nΔ moda vs. sugerido: {signo}{distancia}"
    #     else:
    #         titulo_completo = nombre_largo
    #     ax.set_title(titulo_completo, fontsize=10)

    #     ax.set_xlabel("Cuatrimestre Cursado (step_nro)", fontsize=8)
    #     ax.set_ylabel("Estudiantes Únicas", fontsize=8)
    #     ax.set_xticks(counts["step_nro"])

    #     if cuatri_sugerido is not None:
    #         ax.bar(0, 0, color=ROJO, label=f"Cuatrimestre sugerido ({cuatri_sugerido})")
    #         ax.legend(loc="upper right", fontsize=9)

    #     ax.tick_params(labelsize=8)
    #     ax.grid(axis="y", linestyle="--", alpha=0.4)

    for ax, codigo in zip(axes, codigos):
        subset = df_con_step[df_con_step[materia_column] == codigo]
        cuatri_sugerido = cuatri_map.get(codigo)
        nombre_largo = nombre_map.get(codigo, codigo)

        if subset.empty or cuatri_sugerido is None or pd.isna(cuatri_sugerido):
            ax.set_title(f"{nombre_largo}\n(Sin datos o sin cuatri sugerido)", fontsize=10)
            ax.axis("off")
            continue

        # 1. Calcular la mediana empírica (estudiante central)
        # Se agrupa por persona y se toma el primer intento (min) para no sobrerrepresentar recursantes
        mediana_empirica = subset.groupby(persona_column)["step_nro"].min().median()

        # 2. Calcular el Delta (Diferencia)
        delta = mediana_empirica - cuatri_sugerido

        # 3. Determinar la categoría según el Delta
        if delta < 0:
            estado = "Adelantamiento"
        elif delta == 0:
            estado = "Sincronía"
        else:
            estado = "Retraso estructural"

        # Construir los conteos para las barras del histograma
        counts = build_materia_step_counts(df_con_step, codigo, persona_column, materia_column)
        
        colors = [
            ROJO if step == cuatri_sugerido else AZUL
            for step in counts["step_nro"]
        ]
        ax.bar(counts["step_nro"], counts["cantidad_personas"], color=colors, edgecolor="black")

        # 4. Trazar la línea de la mediana
        ax.axvline(x=mediana_empirica, color="green", linestyle="--", linewidth=2, label=f"Mediana: {mediana_empirica:.0f}")

        # 5. Actualizar el título para mostrar el Cuatri Sugerido, la Mediana, el Delta y el Estado
        titulo_completo = (
            f"{nombre_largo}\n"
            f"Sugerido: {int(cuatri_sugerido)} | Mediana: {mediana_empirica:.0f}\n"
            f"$\Delta$ = {delta:.0f} ({estado})"
        )
        if cuatri_sugerido is not None:
            ax.bar(0, 0, color=ROJO, label=f"Cuatrimestre sugerido ({cuatri_sugerido})")
            ax.legend(loc="upper right", fontsize=9)

        ax.set_title(titulo_completo, fontsize=10)
        
        ax.set_xlabel("Cuatrimestre Cursado (step_nro)", fontsize=8)
        ax.set_ylabel("Estudiantes Únicas", fontsize=8)
        ax.set_xticks(counts["step_nro"])
        ax.tick_params(labelsize=8)
        ax.grid(axis="y", linestyle="--", alpha=0.4)
        ax.legend(loc="upper right", fontsize=8)

    # fig.suptitle(titulo_plot, fontsize=12*(n_columnas/2), y=1)
    # ocultar ejes sobrantes si la grilla no se completa exactamente
    for ax in axes[n_materias:]:
        ax.axis("off")

    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    return fig


def plot_todas_las_materias(
    df_con_step: pd.DataFrame,
    plan: pd.DataFrame,
    output_path: Path,
    persona_column: str = "persona",
    materia_column: str = "codigo_materia",
) -> plt.Figure:
    """Genera UNA sola imagen con los histogramas de todas las materias del
    plan CSV, ordenados en una grilla de 4 columnas."""
    codigos = sorted(plan["codigo_materia"].dropna().unique())
    return _plot_en_grilla(df_con_step, plan, codigos, n_columnas=3, output_path=output_path,titulo_plot='Distribución de Nro. Cuatrimestre en las Cursadas de cada Materia',
                            persona_column=persona_column, materia_column=materia_column)


def plot_materias_seleccionadas(
    df_con_step: pd.DataFrame,
    plan: pd.DataFrame,
    codigos_seleccionados: list[str],
    output_path: Path,
    persona_column: str = "persona",
    materia_column: str = "codigo_materia",
    n_columnas: int = 2
) -> plt.Figure:
    """Igual que plot_todas_las_materias pero restringido a un subconjunto de
    códigos de materia dado, en una grilla de 2 columnas."""
    return _plot_en_grilla(df_con_step, plan, codigos_seleccionados, n_columnas=n_columnas, output_path=output_path,titulo_plot='Distribución de Nro. Cuatrimestre en las Cursadas de cada Materia (Materias Representativas)',
                            persona_column=persona_column, materia_column=materia_column)


In [ ]:
import os
BASE_DIR = os.getcwd()
PLANS_DIR = Path(BASE_DIR) / "visualization_prod" / "plan"
PLANS_DIR

In [ ]:
if __name__ == "__main__":
    # --- Ajustar paths según ubicación real de los archivos ---
    SOURCE_PATH = Path("bases/para tesis LCC_v3.xlsx")
    PLAN_CSV_PATH = Path("visualization_prod/plan/LCCOMP_1993.csv")

    ACADEMIC_DATASET  = cargar_base_siu(SOURCE_PATH) #dataset
    YEAR_LIST = range(2010,2026)

    
    CODIGO_CARRERA = "LCCOMP"
    PLAN_VALUE = "1993"
    PLAN_ID = f"{CODIGO_CARRERA}_{PLAN_VALUE}"
    parsed_plan = load_plan_from_directory(PLANS_DIR, PLAN_ID)
    catalog = build_unified_catalog({PLAN_ID: parsed_plan})

    must_be_1st = get_first_year_materias(pd.DataFrame(catalog), plan_id=PLAN_ID, max_cuatri=2)
    
    camada = get_camada(
        ACADEMIC_DATASET.planes_por_alumno,
        ACADEMIC_DATASET.cursadas_por_alumno,
        year_list=YEAR_LIST,
        plan=PLAN_VALUE,
        codigo_carrera="LCCOMP",
        must_be_1st_materias=must_be_1st,
    )
    df_con_step = compute_step_nro_por_cursada(ACADEMIC_DATASET.cursadas_por_alumno, camada=camada)
    plan = load_plan_csv(PLAN_CSV_PATH)

    plot_todas_las_materias(
        df_con_step,
        plan,
        output_path=Path("../assets_latex/histogramas_output/todas_las_materias.png"),
    )

    plot_materias_seleccionadas(
        df_con_step,
        plan,
        codigos_seleccionados=["COMP930001", "COMP930009","COMP930012","COMP930008"],
        output_path=Path("../assets_latex/histogramas_output/materias_seleccionadas.png"),
    )

    mats_1  = [ "COMP930001",
                "COMP930003",
                "COMP930004",
                "COMP930005",
                "COMP930007",
                "COMP930008",
                "COMP930009",
                "COMP930010",
                "COMP930011"]
    
    plot_materias_seleccionadas(
        df_con_step,
        plan,
        codigos_seleccionados=mats_1,
        output_path=Path("../assets_latex/histogramas_output/todas_las_materias_1.png"),
        n_columnas = 3
    )
    

    mats_2 = [  "COMP930012",
                "COMP930013",
                "COMP930014",
                "COMP930015",
                "COMP930016",
                "COMP930017",
                "COMP930018",
                "COMP930019"]

    plot_materias_seleccionadas(
        df_con_step,
        plan,
        codigos_seleccionados=mats_2,
        output_path=Path("../assets_latex/histogramas_output/todas_las_materias_2.png"),
        n_columnas = 3
    )